1. Scenario:  Enterpise with 100s of services; developer says, "I'm deploying 3.4 of the Payment API".
2. AI agent:  "What could this break?"
3. The key difference is not traversing towards a root cause, but traversing AWAY from a changed component
4. Dependency graph below

In [1]:
dependency_graph = {
    "payment_api": [
        {"node": "checkout_service", "relationship": "used_by"},
        {"node": "billing_service", "relationship": "used_by"},
    ],

    "checkout_service": [
        {"node": "mobile_app", "relationship": "used_by"},
        {"node": "website", "relationship": "used_by"},
    ],

    "billing_service": [
        {"node": "finance_dashboard", "relationship": "used_by"},
    ],

    "mobile_app": [],
    "website": [],
    "finance_dashboard": [],
}

## State Invariants

- The queue is first in, first out (FIFO).
- Nodes are explored in order of increasing distance from the start.
- A node is marked as visited when it is queued.
- Each queued path describes how its node was reached.
- The first path found to the target has the fewest relationships.


In [12]:
from collections import deque


def change_impact_analysis(graph, start):
    queue = deque()
    visited = {start}
    impacted = []

    queue.append((start, []))

    while queue:
        current_node, dependency_path = queue.popleft()

        for edge in graph.get(current_node, []):
            neighbor_node = edge["node"]
            relationship = edge["relationship"]

            if neighbor_node not in visited:
                visited.add(neighbor_node)

                new_relationship = {
                    "from": current_node,
                    "relationship": relationship,
                    "to": neighbor_node,
                }

                new_path = dependency_path + [new_relationship]

                impacted.append({
                    "service": neighbor_node,
                    "depth": len(new_path),
                    "dependency_path": new_path,
                })

                queue.append((neighbor_node, new_path))

    return impacted

In [15]:
change_impact_analysis(dependency_graph, start="payment_api")

discovered neighbor node checkout_service through payment_api ----> used_by ----> checkout_service
discovered neighbor node billing_service through payment_api ----> used_by ----> billing_service
discovered neighbor node mobile_app through checkout_service ----> used_by ----> mobile_app
discovered neighbor node website through checkout_service ----> used_by ----> website
discovered neighbor node finance_dashboard through billing_service ----> used_by ----> finance_dashboard


[{'service': 'payment_api', 'depth': 1, 'to': 'checkout_service'},
 {'service': 'payment_api', 'depth': 1, 'to': 'billing_service'},
 {'service': 'checkout_service', 'depth': 2, 'to': 'mobile_app'},
 {'service': 'checkout_service', 'depth': 2, 'to': 'website'},
 {'service': 'billing_service', 'depth': 2, 'to': 'finance_dashboard'}]